# import libraries

In [328]:
import numpy as np
import pandas as pd
import re
import nltk
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as stop_words
from nltk.stem import WordNetLemmatizer
# Ensure you have the necessary NLTK data files for lemmatization
nltk.download('wordnet')
nltk.download('omw-1.4')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression ## Logistic Regression
from xgboost import XGBClassifier                     # XGBoost
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from wordcloud import WordCloud, STOPWORDS
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


# load file

In [330]:
data=pd.read_csv("C:/Users/DELL/nlp/Sentimentdataset/sentimentdataset.csv")

# shape`

In [332]:
data.shape

(732, 15)

# data

In [334]:
data

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
727,728,732,Collaborating on a science project that receiv...,Happy,2017-08-18 18:20:00,ScienceProjectSuccessHighSchool,Facebook,#ScienceFairWinner #HighSchoolScience,20.0,39.0,UK,2017,8,18,18
728,729,733,Attending a surprise birthday party organized ...,Happy,2018-06-22 14:15:00,BirthdayPartyJoyHighSchool,Instagram,#SurpriseCelebration #HighSchoolFriendship,25.0,48.0,USA,2018,6,22,14
729,730,734,Successfully fundraising for a school charity ...,Happy,2019-04-05 17:30:00,CharityFundraisingTriumphHighSchool,Twitter,#CommunityGiving #HighSchoolPhilanthropy,22.0,42.0,Canada,2019,4,5,17
730,731,735,"Participating in a multicultural festival, cel...",Happy,2020-02-29 20:45:00,MulticulturalFestivalJoyHighSchool,Facebook,#CulturalCelebration #HighSchoolUnity,21.0,43.0,UK,2020,2,29,20


# selecting 2 columns

In [336]:
# Selecting multiple columns (e.g., 'col1', 'col2', 'col
data_df= data[['Text','Sentiment']]


# info

In [338]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Text       732 non-null    object
 1   Sentiment  732 non-null    object
dtypes: object(2)
memory usage: 11.6+ KB


# describe

In [340]:
data_df.describe()

,Text,Sentiment
count,732,732
unique,707,279
top,"A compassionate rain, tears of empathy fallin...",Positive
freq,3,44


# sum of null

In [342]:
data_df.isnull().sum()

Text         0
Sentiment    0
dtype: int64

# duplicates

In [344]:
data_df.duplicated().sum()

24

In [345]:
data_df=data_df.drop_duplicates()

In [346]:
data_df.duplicated().sum()

0

# info

In [348]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 708 entries, 0 to 731
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Text       708 non-null    object
 1   Sentiment  708 non-null    object
dtypes: object(2)
memory usage: 16.6+ KB


# unique

In [350]:
data_df.nunique()

Text         707
Sentiment    279
dtype: int64

In [351]:
import re
import nltk
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as stop_words
from nltk.stem import WordNetLemmatizer

# lemmatizer

In [353]:
lemmatizer = WordNetLemmatizer()

In [354]:
def clean_text(text):
    # Remove special characters and replace them with spaces
    text = re.sub(r'\W', ' ', text)

    # Convert all characters to lowercase
    text = text.lower()

    # Tokenize the text (split it into individual words)
    words = text.split()

    # Remove stop words and lemmatize each word
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    # Join the cleaned words back into a single string
    return ' '.join(cleaned_words)


In [355]:
data_df['cleaned_text'] = data_df['Text'].apply(clean_text)

# vectorizer

In [357]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize a TF-IDF vectorizer with a maximum of 500 features (terms)
tfidf_vectorizer = TfidfVectorizer(max_features=300)

# Apply the TF-IDF vectorizer to the cleaned email text data and transform it into numerical featur
X= tfidf_vectorizer.fit_transform(data_df['cleaned_text'])

In [358]:
X

<708x300 sparse matrix of type '<class 'numpy.float64'>'
	with 2556 stored elements in Compressed Sparse Row format>

In [359]:
X = X
Y= data_df.Sentiment.to_frame()
Y

,Sentiment
0,Positive
1,Negative
2,Positive
3,Positive
4,Neutral
...,...
727,Happy
728,Happy
729,Happy
730,Happy


# split

In [361]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.30, random_state = 42)

In [362]:
print('Shape of X_train:', X_train.shape)
print('Shape of X_test:', X_test.shape)
print('Shape of y_train:', y_train.shape)
print('Shape of y_test:', y_test.shape)

Shape of X_train: (495, 300)
Shape of X_test: (213, 300)
Shape of y_train: (495, 1)
Shape of y_test: (213, 1)


In [363]:
y_train

,Sentiment
420,Heartbreak
642,Relief
66,Acceptance
11,Negative
254,Playful
...,...
71,Confusion
106,Kind
270,Empathetic
459,Loss


# labelencoder

In [365]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.fit_transform(y_test)

In [366]:
y_train = pd.DataFrame(y_train)
y_test = pd.DataFrame(y_test)

In [367]:
y_train

,0
0,128
1,197
2,0
3,169
4,181
...,...
490,47
491,154
492,84
493,158


In [368]:
y_test

,0
0,58
1,127
2,88
3,33
4,78
...,...
208,99
209,99
210,99
211,81


# SMOTE

# model

# LogisticRegression

In [372]:
from sklearn.linear_model import LogisticRegression ## Logistic Regression
clf_logreg = LogisticRegression()
clf_logreg.fit(X_train, y_train)
# Prediction using Logistic regression
y_pred_logreg = clf_logreg.predict(X_test)
# score
from sklearn.metrics import classification_report
clf_rpt_logreg = classification_report(y_test, y_pred_logreg)
print('classification report for logistic regression \n',clf_rpt_logreg )
from sklearn.metrics import confusion_matrix
print('Confusion matrix for Logistic Regression \n', confusion_matrix(y_test, y_pred_logreg))
# confusion matric
#from sklearn.metrics import confusion_matrix
#print('Confusion matrix for Logistic Regression \n', sns.heatmap(confusion_matrix(y_test, y_pred_logreg), annot = True))

classification report for logistic regression 
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      0.50      0.67         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00         4
          11       0.00      0.00      0.00         2
          12       0.00      0.00      0.00         1
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00         1
          16       0.00      0.00

# RandomForestClassifier

In [374]:
from sklearn.ensemble import RandomForestClassifier
# Creating a Random Forest Classifier
rf_classifier = RandomForestClassifier()
# Fitting the classifier to the training data
rf_classifier.fit(X_train, y_train)
# Making predictions using the Random Forest classifier
y_pred_rf = rf_classifier.predict(X_test)
# Generating a classification report for the Random Forest classifier
from sklearn.metrics import classification_report
classification_report_rf = classification_report(y_test, y_pred_rf)
# Printing the classification report
print('Classification report for Random Forest Classifier:\n', classification_report_rf)

Classification report for Random Forest Classifier:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.67      1.00      0.80         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00         4
          11       0.00      0.00      0.00         2
          12       0.00      0.00      0.00         1
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00         1
          16       0.00     

# KNeighborsClassifier

In [376]:
from sklearn.neighbors import KNeighborsClassifier
# Creating a K-Nearest Neighbors Classifier
knn_classifier = KNeighborsClassifier()
# Fitting the classifier to the training data
knn_classifier.fit(X_train, y_train)
# Making predictions using the K-Nearest Neighbors classifier
y_pred_knn = knn_classifier.predict(X_test)
# Generating a classification report for the K-Nearest Neighbors classifier
from sklearn.metrics import classification_report
classification_report_knn = classification_report(y_test, y_pred_knn)
# Printing the classification report
print('Classification report for K-Nearest Neighbors Classifier:\n', classification_report_knn)

# confusion matric
#from sklearn.metrics import confusion_matrix
#print('Confusion matrix for K-Nearest Neighbors Classifier \n', sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot = True))

Classification report for K-Nearest Neighbors Classifier:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.67      1.00      0.80         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00         4
          11       0.00      0.00      0.00         2
          12       0.00      0.00      0.00         1
          13       0.01      0.50      0.02         2
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00         1
          16       0.0

# svc

In [378]:
from sklearn.svm import SVC
# Creating a Support Vector Classifier
svc_classifier = SVC()
# Fitting the classifier to the training data
svc_classifier.fit(X_train, y_train)
# Making predictions using the Support Vector Classifier
y_pred_svc = svc_classifier.predict(X_test)
# Generating a classification report for the Support Vector Classifier
from sklearn.metrics import classification_report
classification_report_svc = classification_report(y_test, y_pred_svc)
# Printing the classification report
print('Classification report for Support Vector Classifier:\n', classification_report_svc)

# confusion matric
#from sklearn.metrics import confusion_matrix
#print('Confusion matrix for Support Vector Classifier \n', sns.heatmap(confusion_matrix(y_test, y_pred_svc), annot = True))

Classification report for Support Vector Classifier:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.50      0.50      0.50         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00         4
          11       0.00      0.00      0.00         2
          12       0.00      0.00      0.00         1
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00         1
          16       0.00    

# DecisionTreeClassifier

In [380]:
from sklearn.tree import DecisionTreeClassifier
# Creating a Decision Tree Classifier
dt_classifier = DecisionTreeClassifier()
# Fitting the classifier to the training data
dt_classifier.fit(X_train, y_train)
# Making predictions using the Decision Tree Classifier
y_pred_dt = dt_classifier.predict(X_test)
# Generating a classification report for the Decision Tree Classifier
from sklearn.metrics import classification_report
classification_report_dt = classification_report(y_test, y_pred_dt)
# Printing the classification report
print('Classification report for Decision Tree Classifier:\n', classification_report_dt)

Classification report for Decision Tree Classifier:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.67      1.00      0.80         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00         4
          11       0.00      0.00      0.00         2
          12       0.00      0.00      0.00         1
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00         1
          16       0.00     

In [416]:
from sklearn.ensemble import GradientBoostingClassifier
# Creating a Gradient Boosting Classifier
from sklearn.metrics import classification_report, accuracy_score
gb_classifier = GradientBoostingClassifier()
# Fitting the classifier to the training data
gb_classifier.fit(X_train, y_train)
# Making predictions using the Gradient Boosting Classifier
y_pred_gb = gb_classifier.predict(X_test)
classification_report_gb = classification_report(y_test, y_pred_gb)
accuracy_gb = accuracy_score(y_test, y_pred_gb)
# Print the results
print("gb Accuracy:", accuracy_gb)
print('Classification report for Gradient Boosting Classifier:\n', classification_report_gb)
print('Confusion matrix for Gradient Classifier \n', confusion_matrix(y_test, y_pred_gb))

gb Accuracy: 0.009389671361502348
Classification report for Gradient Boosting Classifier:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.67      1.00      0.80         2
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00         4
          11       0.00      0.00      0.00         2
          12       0.00      0.00      0.00         1
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00 